# Part 1 — Reinforcement Learning overview

This notebook gives the big RL map before individual algorithms. RL is learning by interaction: an agent observes state, takes action, receives reward, and improves a policy.

**Learning style:** mechanisms first → frameworks second → real systems third. The notebook is intentionally slow, explicit, and beginner-friendly.

In [ ]:
# Setup: run this first.
# Works from the repository root. In Colab, clone the repo first, then run from inside it.
from pathlib import Path
import sys, math, random
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    print('Tip: run this notebook from the repository root, or clone the repo in Colab first.')
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## Notebook type and ordered study structure

**Notebook type:** Type A — core mechanism notebook. This should be studied slowly because the mechanism appears again and again across the whole repo.

**Your objective in this notebook:** Understand the RL problem before algorithms: state, action, reward, return, policy, and value.

Use this exact order every time:

1. **Why this matters** — identify the real problem this topic solves.
2. **Mental model** — explain the idea in plain language before symbols.
3. **Math mechanism** — write the smallest formula and define every symbol.
4. **From-scratch code** — run/read the simple implementation slowly.
5. **Line-by-line explanation** — trace inputs, internal variables, update rule, and outputs.
6. **Debug/visualize** — print shapes, values, curves, maps, or weights so behavior is visible.
7. **Framework version** — map the mechanism to a real library/API.
8. **Scratch → framework mapping** — write what the framework hides and what it exposes.
9. **Real-system role** — place the topic inside robotics, driving, drones, manipulation, or VLA.
10. **Failure modes** — list how it breaks and how you would notice.
11. **Exercises** — change parameters, break the example, and explain the result.
12. **Mini-project** — build a small artifact you can keep.
13. **Next step** — choose the next notebook or tool.

**Mental model for this topic:** RL is delayed-credit learning: today’s action changes tomorrow’s data and reward.

**Core math / mechanism to keep in mind:** Return G_t = r_t + gamma r_{t+1} + gamma^2 r_{t+2}+...

**Recommended debug habit:** after every code cell, ask “what are the inputs, what changed, and what would be unsafe or wrong in a real robot?”

## From-scratch focus and code-reading checklist

**Scratch focus:** Tiny two-state MDP and Q-table update.

When reading code in this notebook or the matching repo module, trace it like this:

| Step | Question to answer |
|---|---|
| Input | What is the state, observation, tensor, point, reward, or measurement? |
| Representation | Is it a scalar, vector, matrix, image, point cloud, token sequence, or action? |
| Mechanism | Which line implements the math/update rule? |
| Parameters | Which numbers are hyperparameters, physical constants, or learned weights? |
| Output | What changed after the step? |
| Debug signal | What should I print/plot to know it is working? |

Do not move to the framework/API version until you can explain the scratch version without reading the code comments.

## Visualization and debugging ideas

Use at least one of these while studying:

- Print tensor/vector shapes before and after the core operation.
- Print the first few values before and after an update.
- Plot a curve when there is learning, control, filtering, planning, or optimization.
- Draw frames, maps, paths, sensor rays, or attention matrices when geometry is involved.
- Change one parameter at a time and predict the effect before running.

For this notebook, a useful first visualization/debug target is: **Tiny two-state MDP and Q-table update.**

## Scratch → framework mapping

| From-scratch idea in this repo | Practical framework/API | What to learn from the framework |
|---|---|---|
| custom env loop | Gymnasium Env API | Gymnasium standardizes reset/step/action_space/observation_space |
| manual Q table | Stable-Baselines3 algorithms | frameworks replace update loops with tested implementations |
| single env | vectorized envs | production training gathers many rollouts in parallel |

**Framework learning rule:** do not memorize the API first. First identify which scratch concept it replaces, then learn its inputs, outputs, configuration, and failure modes.

## Real-system application

Robots, cars, and drones use RL when actions affect future situations and hand-written controllers/rewards are insufficient.

Ask these system questions:

1. What module produces the input to this component?
2. What module consumes its output?
3. What latency, safety, calibration, or data-format assumptions exist?
4. What metric tells me this component is good enough for the larger system?

## Failure modes and debugging

Common ways this topic can fail:

- reward hacking
- unsafe exploration
- non-Markov observations
- sim overfitting

For each failure, write:

- **Symptom:** what would I see in logs, plots, robot behavior, or evaluation?
- **Likely cause:** what assumption broke?
- **First debug action:** what is the smallest thing to inspect?

## Mini-project and mastery checklist

**Mini-project:** Define an MDP for one target application: car lane keeping, drone hover, or robot reaching.

Mastery checklist:

- [ ] I can explain the mental model in one paragraph.
- [ ] I can write the core formula and define every symbol.
- [ ] I can run or read the scratch code and point to the core update/operation.
- [ ] I can name the production framework/API version of the same idea.
- [ ] I can describe where this topic sits in a robot/car/drone/VLA stack.
- [ ] I can name at least three failure modes and one debug action for each.

**Next study steps:** RL 00 foundations, RL 01 tabular, Part 12 imitation learning

## 1. Mental model

This notebook gives the big RL map before individual algorithms. RL is learning by interaction: an agent observes state, takes action, receives reward, and improves a policy.

Before code, write one sentence in your own words: *what problem does this topic solve?*

## 2. Mechanism and math

The Markov Decision Process is the base object:
\[
(S, A, P, R, \gamma)
\]
State `s`, action `a`, transition probability `P(s'|s,a)`, reward `R`, and discount `gamma`. The core quantity is return:
\[
G_t = r_t + \gamma r_{t+1}+\gamma^2 r_{t+2}+...\]
Value functions estimate expected return; policies choose actions.

## 3. From-scratch lab

Run a tiny hand-made environment so the interaction loop is visible before using Gymnasium or deep RL.

Read every line. The code avoids clever abstractions so you can see the mechanism.

In [ ]:
# A two-state MDP: start at 0. Action 1 goes to goal with reward; action 0 wastes time.
state = 0
Q = [[0.0, 0.0], [0.0, 0.0]]
alpha, gamma, eps = 0.3, 0.9, 0.2
import random

for episode in range(200):
    state = 0
    for t in range(5):
        action = random.randrange(2) if random.random() < eps else max(range(2), key=lambda a: Q[state][a])
        if state == 0 and action == 1:
            next_state, reward, done = 1, 1.0, True
        else:
            next_state, reward, done = state, 0.0, False
        target = reward + gamma * max(Q[next_state]) * (not done)
        Q[state][action] += alpha * (target - Q[state][action])
        state = next_state
        if done: break
print('Q table:', Q)
print('best action at start:', max(range(2), key=lambda a: Q[0][a]))

## 3.1 Code reading guide

When you read the previous cell, do not treat it as a black box. Trace it in this order:

1. **Inputs:** what are the given numbers, observations, states, rewards, or measurements?
2. **Internal variables:** what does each variable represent physically or mathematically?
3. **Update rule:** which line is the core mechanism from the math section?
4. **Output:** what should change if the mechanism is working?
5. **Failure case:** what parameter could make the example unstable, wrong, or unsafe?

This habit is the bridge between toy examples and real robotics code: every simulator, ROS node, policy, controller, or perception model still has inputs, state, an update rule, and outputs.

## 4. Framework/practice view

Frameworks supply environments, vectorization, logging, and tuned algorithms. Gymnasium standardizes `reset` and `step`; Stable-Baselines3 gives production-ready PPO/SAC/DQN; RLlib scales to distributed training.

The goal is not to replace understanding with APIs. The goal is to recognize the same mechanism when a library hides the details.

In [ ]:
try:
    import gymnasium as gym
    env = gym.make('CartPole-v1')
    obs, info = env.reset(seed=0)
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, info = env.step(action)
    print('obs shape:', obs.shape, 'sample action:', action, 'reward:', reward)
    env.close()
except ModuleNotFoundError as e:
    print('Install gymnasium to run this:', e)

## 4.1 Framework comparison checklist

After running or reading the framework cell, write a small mapping table for yourself:

| Question | Your answer |
|---|---|
| What object/function in the framework replaces the scratch code? |  |
| Which parameters match the math symbols? |  |
| What details does the framework hide? |  |
| What new engineering concerns appear? | installation, devices, logging, data formats, batching, safety, versioning |

This is where top-down learning becomes useful: you learn the professional API **without losing the mechanism**

## 5. Real-system connection

Real systems use RL when actions affect future data: robot locomotion, drone aggressive flight, simulator-trained autonomous-driving policies, or RL fine-tuning after imitation learning. Always ask: what is the state, action, reward, reset condition, and safety constraint?

## 6. Exercises

1. For a drone landing task, define state/action/reward/done.
2. For a robot arm reaching task, define a bad reward and a better reward.
3. Explain why offline logs alone are imitation/offline RL, not normal online RL.

**Notebook habit:** after each exercise, add a short note explaining what changed and why it matters in a robot/car/drone/VLA stack.